In [ ]:
pip install streamlit python-dotenv langchain langchain-community langchain-openai pypdf faiss-cpu qdrant-client

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
MODEL_NAME = os.getenv("MODEL_NAME")

In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

Saving agreemnent.pdf to agreemnent (1).pdf


A basic Legal Document Analyzer, but it is not using embeddings, chunking, FAISS, or RAG

WORKFLOW                                      
PDF -> PyPDFLoader -> Extract Entire Text - >Send Whole Document to LLM -> Analysis

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
import os

agreement_type = input("Enter agreement type: ")

loader = PyPDFLoader(pdf_path)
pages = loader.load()

agreement_text = "\n".join([page.page_content for page in pages])

llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"]
)

prompt = f"""
You are an expert legal document analyzer.
Analyze this document as a {agreement_type}.
Document:
{agreement_text}
Return:
1. Document summary
2. Parties involved
3. Important clauses
4. Risky clauses
5. Missing clauses
6. Obligations of each party
7. Payment or penalty terms if any
8. Confidentiality terms if any
9. Termination conditions
10. Final risk rating: Low, Medium, or High
11. Practical recommendation
Do not give legal advice. Only provide an educational document analysis.
"""
response = llm.invoke(prompt)
print(response.content)

Enter agreement type: employee
**1. Document Summary:**  
This Employment Agreement establishes the terms of employment between Synergy Resources Corporation (the "Company") and Ed Holloway (the "Employee"). The agreement outlines the Employee's role, compensation, duration of employment, termination conditions, confidentiality obligations, and indemnification provisions.

**2. Parties Involved:**  
- **Company:** Synergy Resources Corporation, a Colorado corporation.  
- **Employee:** Ed Holloway.

**3. Important Clauses:**  
- **Employment Role:** Employee serves as President and Chief Executive Officer, devoting approximately 80% of his time to the Company.
- **Compensation (Sec. 3):** Base salary is set at $420,000 annually, with bonuses tied to the successful drilling of wells.
- **Vacation and Benefits (Sec. 3.3 - 3.4):** Employee entitled to participate in Company benefits and eight weeks of paid vacation annually.
- **Confidential Information Agreement (Sec. 5):** Obligates the

RAG (Retrieval-Augmented Generation) pipeline using FAISS.

Overall Workflow                                            
PDF -> Load Pages -> Chunking -> Embeddings -> FAISS Vector Store -> User Question -> Question Embedding -> Top-K Similarity Search -> Retrieve Relevant Chunks -> LLM -> Answer

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120
)

chunks = splitter.split_documents(pages)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"]
)

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
question = input("Ask a question from the document: ")
retrieved_docs = retriever.invoke(question)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])
rag_prompt = f"""
You are a document analysis assistant.
Answer only using the given context.
Context:
{context}
Question:
{question}
Answer:
"""

rag_response = llm.invoke(rag_prompt)
print(rag_response.content)

Qdrant: operations on vector database

In [ ]:
pip install qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 2.5 MB/s eta 0:00:00


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Document
from google.colab import userdata
import os

os.environ["QDRANT_API_KEY"] = userdata.get("qdrant")

# connect to Qdrant Cloud
client = QdrantClient(
    url="https://0f360b1f-6bb3-497f-ac4a-0777b095ed2b.sa-east-1-0.aws.cloud.qdrant.io",
    api_key=os.environ["QDRANT_API_KEY"],
    cloud_inference=True
)

In [ ]:
# create collection
client.create_collection(
    collection_name="items",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

True

In [ ]:
menu_items = [
    ("Pad Thai with Tofu", "Stir-fried rice noodles with tofu bean sprouts scallions and crushed peanuts in traditional tamarind sauce", "$13.95", "Noodles"),
    ("Grilled Salmon Fillet", "Wild-caught Atlantic salmon grilled with lemon butter and fresh herbs served with seasonal vegetables", "$24.50", "Seafood Entrees"),
    ("Mushroom Risotto", "Creamy arborio rice with mixed mushrooms parmesan truffle oil and fresh thyme", "$16.75", "Vegetarian"),
    ("Bibimbap Bowl", "Korean rice bowl with seasoned vegetables fried egg gochujang sauce and choice of protein", "$14.50", "Korean Bowls"),
    ("Falafel Wrap", "Crispy chickpea fritters with hummus tahini cucumber tomato and pickled vegetables in warm pita", "$11.25", "Mediterranean"),
    ("Shrimp Tacos", "Three soft tacos with grilled shrimp cabbage slaw chipotle aioli and fresh lime", "$13.00", "Tacos"),
    ("Vegetable Curry", "Mixed vegetables in aromatic coconut curry sauce with jasmine rice and naan bread", "$12.95", "Indian Curries"),
    ("Tuna Poke Bowl", "Fresh ahi tuna with avocado edamame cucumber seaweed salad over sushi rice with spicy mayo", "$16.50", "Poke Bowls"),
    ("Margherita Pizza", "Fresh mozzarella san marzano tomatoes basil and extra virgin olive oil on wood-fired crust", "$14.00", "Pizza"),
    ("Chicken Tikka Masala", "Tandoori chicken in creamy tomato sauce with aromatic spices served with basmati rice", "$15.95", "Indian Entrees"),
    ("Greek Salad", "Romaine lettuce tomatoes cucumbers kalamata olives feta cheese red onion with lemon oregano dressing", "$10.50", "Salads"),
    ("Lobster Roll", "Fresh Maine lobster meat with light mayo on toasted buttery roll served with chips", "$22.00", "Seafood Sandwiches"),
    ("Quinoa Buddha Bowl", "Organic quinoa with roasted chickpeas kale sweet potato tahini dressing and hemp seeds", "$13.50", "Healthy Bowls"),
    ("Beef Pho", "Traditional Vietnamese beef noodle soup with rice noodles fresh herbs bean sprouts and lime", "$12.75", "Noodle Soups"),
    ("Eggplant Parmesan", "Breaded eggplant layered with marinara mozzarella and parmesan served with pasta", "$15.25", "Italian Entrees"),
    ("Crab Cakes", "Maryland-style lump crab cakes with remoulade sauce and mixed greens", "$18.50", "Seafood Appetizers"),
    ("Tofu Stir Fry", "Crispy tofu with broccoli bell peppers snap peas in garlic ginger sauce over steamed rice", "$12.50", "Vegetarian Entrees"),
    ("Salmon Sushi Platter", "12 pieces of fresh salmon nigiri and sashimi with wasabi pickled ginger and soy sauce", "$19.95", "Sushi"),
    ("Caprese Sandwich", "Fresh mozzarella tomatoes basil pesto balsamic glaze on ciabatta bread", "$11.75", "Sandwiches"),
    ("Tom Yum Soup", "Spicy and sour Thai soup with shrimp lemongrass galangal mushrooms and kaffir lime leaves", "$11.50", "Soups"),
    ("Lentil Dal", "Red lentils simmered with turmeric cumin coriander served with rice and naan", "$11.95", "Vegan Entrees"),
    ("Fish and Chips", "Beer-battered cod with crispy fries malt vinegar and tartar sauce", "$16.00", "British Classics"),
    ("Veggie Burger", "House-made black bean and quinoa patty with avocado sprouts tomato on brioche bun", "$13.25", "Burgers"),
    ("Miso Ramen", "Rich miso broth with ramen noodles soft-boiled egg bamboo shoots nori and scallions", "$14.50", "Ramen"),
    ("Stuffed Bell Peppers", "Roasted bell peppers filled with rice vegetables herbs and melted cheese", "$13.75", "Vegetarian Entrees"),
    ("Scallop Risotto", "Pan-seared sea scallops over creamy parmesan risotto with white wine and lemon", "$26.50", "Seafood Specials"),
    ("Spring Rolls", "Fresh rice paper rolls with vegetables tofu rice noodles herbs and peanut dipping sauce", "$8.95", "Appetizers"),
    ("Oyster Po Boy", "Fried oysters with lettuce tomato pickles and remoulade on french bread", "$15.50", "Sandwiches"),
    ("Portobello Mushroom Steak", "Grilled portobello cap marinated in balsamic with roasted vegetables and quinoa", "$14.95", "Vegan Entrees"),
    ("Coconut Shrimp", "Jumbo shrimp breaded in shredded coconut served with sweet chili sauce", "$14.25", "Seafood Appetizers")
]

# points generator
points = []
for i, menu_item in enumerate(menu_items):
    point = PointStruct(
        id=i,
        vector=Document(
            text=f"{menu_item[0]} {menu_item[1]}",
            model="sentence-transformers/all-MiniLM-L6-v2"
        ),
        payload={
            "item_name": menu_item[0],
            "description": menu_item[1],
            "price": menu_item[2],
            "category": menu_item[3],
        }
    )
    points.append(point)

# upsert points to collection
client.upsert(
  collection_name="items",
  points=points,
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

Semantic search

In [ ]:
# generate query embedding
query_text = "vegetarian dishes"

# search for similar menu items
results = client.query_points(
    collection_name="items",
    query=Document(text=query_text, model="sentence-transformers/all-MiniLM-L6-v2"),
    with_payload=True,
    limit=5
)

# print results
for result in results.points:
    print(f"Item: {result.payload.get('item_name', 'N/A')}")
    print(f"Score: {result.score}")
    print(f"Description: {result.payload['description'][:150]}...")
    print(f"Price: {result.payload.get('price', 'N/A')}")
    print("---")

Item: Greek Salad
Score: 0.5085151
Description: Romaine lettuce tomatoes cucumbers kalamata olives feta cheese red onion with lemon oregano dressing...
Price: $10.50
---
Item: Shrimp Tacos
Score: 0.50280845
Description: Three soft tacos with grilled shrimp cabbage slaw chipotle aioli and fresh lime...
Price: $13.00
---
Item: Oyster Po Boy
Score: 0.48535526
Description: Fried oysters with lettuce tomato pickles and remoulade on french bread...
Price: $15.50
---
Item: Stuffed Bell Peppers
Score: 0.48238587
Description: Roasted bell peppers filled with rice vegetables herbs and melted cheese...
Price: $13.75
---
Item: Grilled Salmon Fillet
Score: 0.47120357
Description: Wild-caught Atlantic salmon grilled with lemon butter and fresh herbs served with seasonal vegetables...
Price: $24.50
---


In [ ]:
# create collection
client.create_collection(
    collection_name="reviews",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

True

In [ ]:
reviews = [
    ("Tom", "Amazing explanation of difficult concepts and always uses practical examples"),
    ("Tom", "Very patient with students and willing to repeat topics when needed"),
    ("Tom", "Some advanced topics were not explained clearly"),
    ("Steve", "Strong technical knowledge and good problem-solving skills"),
    ("Steve", "Classes are interactive and keep students engaged"),
    ("Steve", "Can sometimes move too quickly through difficult material"),
    ("Sarah", "Excellent communication skills and very organized lectures"),
    ("Sarah", "Provides detailed feedback that helps students improve"),
    ("Sarah", "Encourages participation and teamwork in class"),
    ("John", "Knowledgeable teacher but can appear strict at times"),
    ("John", "Explains concepts clearly with step-by-step examples"),
    ("John", "Not always available for extra doubt-clearing sessions"),
    ("Emily", "Friendly and approachable teacher who supports every student"),
    ("Emily", "Makes complex topics easy to understand through simple explanations")
]

points = []

for i, review in enumerate(reviews):
    point = PointStruct(
        id=i,
        vector=Document(
            text=f"Teacher: {review[0]}. Review: {review[1]}",
            model="sentence-transformers/all-MiniLM-L6-v2"
        ),
        payload={
            "teacher": review[0],
            "review": review[1]
        }
    )
    points.append(point)

# Upload to Qdrant
client.upsert(
    collection_name="reviews",
    points=points
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

# Create a payload index for the 'teacher' field
client.create_payload_index(
    collection_name="reviews",
    field_name="teacher",
    field_schema="keyword"
)

results = client.scroll(
    collection_name="reviews",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="teacher",
                match=MatchValue(value="Tom")
            )
        ]
    ),
    with_payload=True
)

tom_reviews = []
for point in results[0]:
    tom_reviews.append(point.payload["review"])

print(tom_reviews)

['Amazing explanation of difficult concepts and always uses practical examples', 'Very patient with students and willing to repeat topics when needed', 'Some advanced topics were not explained clearly']


In [ ]:
reviews_text = "\n".join(tom_reviews)
prompt = f"""
Based on the following student reviews, describe what kind of teacher Tom is.
Reviews:
{reviews_text}
Provide a balanced summary.
"""


Based on the following student reviews, describe what kind of teacher Tom is.
Reviews:
Amazing explanation of difficult concepts and always uses practical examples
Very patient with students and willing to repeat topics when needed
Some advanced topics were not explained clearly
Provide a balanced summary.



In [ ]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
os.environ["MODEL_NAME"] = "openai/gpt-4o-mini"

In [ ]:
%pip install -U langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 1.3 MB/s eta 0:00:00


In [ ]:
from langchain_openai import ChatOpenAI
import os
llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"]
)

response = llm.invoke(prompt)
print(response.content)

Based on the student reviews, Tom appears to be a dedicated and effective teacher with strengths in explaining complex concepts through practical examples and a notable patience with his students. His ability to elucidate difficult topics suggests he has a strong command of the material and a talent for making it accessible. Students appreciate his willingness to revisit topics when students struggle, which indicates a supportive and understanding teaching style.

However, it is worth noting that some reviews highlight that certain advanced topics were not explained clearly. This suggests that there may be room for improvement in his approach to more complex subjects, potentially indicating a need for better clarity or alternative methods of explanation in these areas.

Overall, Tom is characterized as a patient and resourceful teacher who is effective in many aspects of his instruction, although he could enhance his effectiveness by focusing on clearer explanations of more advanced to